# Evaluate baselines on the Skolfie corpus

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohaaa825/Skolfie/blob/main/notebooks/evaluate.ipynb)

Roadmap step 3. Uses the corpus that `build_corpus.ipynb` wrote to your Drive and:

1. makes the fixed, versioned evaluation splits (`splits.parquet`, next to the corpus),
2. trains and scores the baselines (length-only and TF-IDF + logistic regression),
3. writes a report card per model to `My Drive/skolfie/reports/<model>/` and a comparison table.

Use **Runtime → Run all**. A CPU runtime is enough. The TF-IDF baseline takes the longest (a few minutes).
Optional Colab secret: `GITHUB_TOKEN` if the repo is private.

## 1. Settings

In [ ]:
DATA_DIR = "/content/drive/MyDrive/skolfie/data/processed"
REPORTS_DIR = "/content/drive/MyDrive/skolfie/reports"

REPO = "mohaaa825/Skolfie"
BRANCH = "main"

MODELS = ["length", "tfidf"]  # baselines to evaluate
MAX_TRAIN = None              # e.g. 50000 to train faster on a sample; None = full train split
MAX_EVAL = None               # e.g. 20000 to cap each test split; None = everything

## 2. Mount Drive, clone and install

In [ ]:
import os
import subprocess

from google.colab import drive

drive.mount("/content/drive")
os.makedirs(REPORTS_DIR, exist_ok=True)
assert os.path.exists(f"{DATA_DIR}/corpus.parquet"), "Run build_corpus.ipynb first."


def secret(name):
    try:
        from google.colab import userdata

        return userdata.get(name)
    except Exception:
        return None


token = secret("GITHUB_TOKEN")
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
if os.path.isdir("/content/Skolfie/.git"):
    subprocess.run(["git", "-C", "/content/Skolfie", "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, "/content/Skolfie"],
                   check=True)

%cd /content/Skolfie
!git log --oneline -1
!pip install -q -e .

## 3. Make the evaluation splits

Deterministic: the same corpus and config always give the same splits (`splits_version`).
Held out from training: GPT-4 and Cohere texts (`test_ood_generator`) and the news domain
(`test_ood_domain`). See `configs/splits.json`.

In [ ]:
!python scripts/make_splits.py --corpus "{DATA_DIR}/corpus.parquet" \
    --config configs/splits.json --out-dir "{DATA_DIR}"

## 4. Train and score the baselines

In [ ]:
extra = []
if MAX_TRAIN:
    extra += ["--max-train", str(MAX_TRAIN)]
if MAX_EVAL:
    extra += ["--max-eval", str(MAX_EVAL)]

for model in MODELS:
    print(f"\n========== {model} ==========")
    cmd = ["python", "scripts/evaluate.py",
           "--corpus", f"{DATA_DIR}/corpus.parquet",
           "--splits", f"{DATA_DIR}/splits.parquet",
           "--model", model,
           "--out-dir", f"{REPORTS_DIR}/{model}", *extra]
    subprocess.run(cmd, check=True)

## 5. Compare

In [ ]:
import glob

reports = sorted(glob.glob(f"{REPORTS_DIR}/*/report.json"))
out = subprocess.run(["python", "scripts/evaluate.py", "--compare", *reports],
                     check=True, capture_output=True, text=True).stdout
print(out)
with open(f"{REPORTS_DIR}/comparison.md", "w") as f:
    f.write(out)

Reports are in `My Drive/skolfie/reports/`. Tell Claude in the project thread when this has
finished; it reads the report cards from Drive directly.